In [3]:
!pip install gensim --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 80.4 MB/s eta 0:00:00


In [4]:
# Все библиотеки будут здесь, так удобнее

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

import gensim.downloader as api
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam

nltk.download('stopwords')

train_df = pd.read_csv('Corona_NLP_train.csv', encoding='latin-1')
test_df = pd.read_csv('Corona_NLP_test.csv', encoding='latin-1')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
# Предобработка текстов и меток

def clean_sentiment(text):
    if 'Positive' in text: return 2 # Positive
    if 'Negative' in text: return 0 # Negative
    return 1 # Neutral

train_df['Sentiment'] = train_df['Sentiment'].apply(clean_sentiment)
test_df['Sentiment'] = test_df['Sentiment'].apply(clean_sentiment)

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # ссылки
    text = re.sub(r'\@\w+|\#','', text) # user и символы хэштегов
    text = re.sub(r'[^\w\s]', '', text) # пунктуация
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

train_df['clean_text'] = train_df['OriginalTweet'].apply(preprocess_text)
test_df['clean_text'] = test_df['OriginalTweet'].apply(preprocess_text)

In [6]:
# Создаем эмбеддинги
# Загрузка предобученной модели
w2v_model = api.load("word2vec-google-news-300")

# Токенизация для нейросети
max_words = 10000
max_len = 100 # Попробую 100, если будет слишком долго то 50

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_df['clean_text'])

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_df['clean_text']), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_df['clean_text']), maxlen=max_len)

# Создаем матрицу весов для слоя Embedding
embedding_matrix = np.zeros((max_words, 300))
for word, i in tokenizer.word_index.items():
    if i < max_words and word in w2v_model:
        embedding_matrix[i] = w2v_model[word]

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [13]:
# Архитектура RNN
model = Sequential([
    Embedding(max_words, 300, weights=[embedding_matrix], input_length=max_len, trainable=False),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train_seq, train_df['Sentiment'],
                    epochs=10,
                    batch_size=64,
                    validation_split=0.1)

# С одним слоем мне показалось лучше, но если будет совсем низкая accuracy можно добавить и второй

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


579/579 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.6088 - loss: 0.8465 - val_accuracy: 0.7393 - val_loss: 0.6435
Epoch 2/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.7430 - loss: 0.6409 - val_accuracy: 0.7643 - val_loss: 0.5955
Epoch 3/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.7771 - loss: 0.5750 - val_accuracy: 0.7682 - val_loss: 0.5923
Epoch 4/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.8004 - loss: 0.5318 - val_accuracy: 0.7923 - val_loss: 0.5450
Epoch 5/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.8130 - loss: 0.5055 - val_accuracy: 0.8081 - val_loss: 0.5179
Epoch 6/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8269 - loss: 0.4712 - val_accuracy: 0.8144 - val_loss: 0.5063
Epoch 7/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.8426 - loss: 0.4425 - val_accuracy: 0.8285 - val_loss: 0.4940
Epoch 8/10
579/579 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8526 - loss: 0.4108 - val_accuracy: 0.

In [14]:
# Оценка и Macro F1
y_pred = np.argmax(model.predict(X_test_seq), axis=-1)
print(classification_report(test_df['Sentiment'], y_pred, target_names=['Negative', 'Neutral', 'Positive']))

f1 = f1_score(test_df['Sentiment'], y_pred, average='macro')
print(f"Итоговый Macro F1: {f1:.4f}")

119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
              precision    recall  f1-score   support

    Negative       0.81      0.82      0.81      1633
     Neutral       0.75      0.68      0.71       619
    Positive       0.81      0.82      0.82      1546

    accuracy                           0.80      3798
   macro avg       0.79      0.78      0.78      3798
weighted avg       0.80      0.80      0.80      3798

Итоговый Macro F1: 0.7819
